In [ ]:
from sklearn.datasets import fetch_openml
data=fetch_openml('mammography',as_frame=True)
X,y = data.data ,data.target
X.shape

In [ ]:
y.value_counts() #不平衡資料


In [ ]:
from sklearn.model_selection import train_test_split
X_train ,X_test,y_train,y_test= train_test_split(X,y=='1',random_state=0)

In [ ]:
from sklearn.model_selection import cross_validate 
from sklearn.linear_model import LogisticRegression
#採用羅吉斯train kflod=10 print mean auc precision
scores=cross_validate(LogisticRegression(), 
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
scores['test_roc_auc'].mean(),scores['test_precision'].mean()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
#採用隨機森林train kflod=10 print mean auc precision
scores=cross_validate(RandomForestClassifier(),
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
scores['test_roc_auc'].mean(),scores['test_precision'].mean()
#雖然上方準確度都很高，但可信度很低，不平衡資料造成偏誤


In [ ]:
from imblearn.under_sampling import RandomUnderSampler 
import numpy as np
rus=RandomUnderSampler(replacement=False) #採用rus
X_train_subsample, y_train_subsample =rus.fit_resample(X_train,y_train)#重新抽樣data
print(X_train.shape)
print(X_train_subsample.shape)
print(np.bincount(y_train_subsample)) 

In [ ]:
from imblearn.under_sampling import RandomUnderSampler 
import numpy as np
rus=RandomUnderSampler(replacement=False) #採用rus
X_train_subsample, y_train_subsample =rus.fit_resample(X_train,y_train)#重新抽樣data
print(X_train.shape)
print(X_train_subsample.shape)
print(np.bincount(y_train_subsample)) 

In [ ]:
from imblearn.pipeline import make_pipeline as make_imb_pipeline #make pipeline 先rus 後羅吉斯
undersample_pipe=make_imb_pipeline(RandomUnderSampler(),LogisticRegression())
scores=cross_validate(undersample_pipe,
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
scores['test_roc_auc'].mean(),scores['test_precision'].mean() #precision 低很多

In [ ]:
undersample_pipe=make_imb_pipeline(RandomUnderSampler(),RandomForestClassifier())#make pipeline 先rus 後隨機森林
scores=cross_validate(undersample_pipe,
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
scores['test_roc_auc'].mean(),scores['test_precision'].mean()

In [ ]:
from imblearn.over_sampling import RandomOverSampler
#採用oversampling 增加資料
ros=RandomOverSampler()
X_train_ovsample, y_train_ovsample =ros.fit_resample(X_train,y_train)
print(X_train.shape)
print(X_train_ovsample.shape)
print(np.bincount(y_train_ovsample))

In [ ]:
oversample_pipe=make_imb_pipeline(RandomOverSampler(),LogisticRegression())#make pipeline 先ros 後羅吉斯
scores=cross_validate(oversample_pipe,
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
scores['test_roc_auc'].mean(),scores['test_precision'].mean()

In [ ]:
oversample_pipe=make_imb_pipeline(RandomOverSampler(),RandomForestClassifier())#make pipeline 先ros 後隨機森林
scores=cross_validate(oversample_pipe,
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
scores['test_roc_auc'].mean(),scores['test_precision'].mean()

In [ ]:
#smote
from imblearn.over_sampling import SMOTE #用smote生成新樣本
import pandas as pd
smote_pipe=make_imb_pipeline(SMOTE(),LogisticRegression()) #一樣make pipeline
scores=cross_validate(smote_pipe,
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
pd.DataFrame(scores)[['test_roc_auc','test_precision']].mean()

In [ ]:
smote_pipe=make_imb_pipeline(SMOTE(),RandomForestClassifier())
scores=cross_validate(smote_pipe,
                     X_train,y_train,cv=10,
                     scoring=('roc_auc','precision'))
pd.DataFrame(scores)[['test_roc_auc','test_precision']].mean()
